<a href="https://colab.research.google.com/github/Camacho-umu/CamachoMu-ozRuiz/blob/main/TD_Diferencias_Temporales_presentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diferencias Temporales (TD): SARSA y Q-Learning

Este notebook cubre el apartado de **Diferencias Temporales (TD)**.
Objetivo práctico:
- Implementar agentes tabulares **SARSA (on-policy)** y **Q-Learning (off-policy)** en Gymnasium.
- Diseñar un **estudio comparativo** con métricas reproducibles y discusión.

Mismos entornos que los utilziados en las comparaciones de
- `FrozenLake-v1` (mapa 4x4 y 8x8), con `is_slippery` desactivado.


## Setup de las semillas


In [1]:
import os
import gc
import torch
import numpy as np
import gymnasium as gym
import random

from dataclasses import dataclass
from collections import defaultdict, deque
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


# Configuración del dispositivo (CPU o GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Liberación de memoria para evitar problemas de consumo en GPU
gc.collect() # Ejecuta el recolector de basura de Python
torch.cuda.empty_cache() # Vacía la caché de memoria en GPU
# Depuración de errores en CUDA
os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # Muestra errores de CUDA en el punto exacto donde ocurren

# Configuración de la semilla para reproducibilidad
seed = 42 # Se define una semilla fija
# Fijar la semilla en NumPy
np.random.seed(seed) # Para generar números aleatorios consistentes en NumPy
np.random.default_rng(seed) # Establece una instancia del generador de NumPy con la misma semilla
# Fijar la semilla en Python
os.environ['PYTHONHASHSEED'] = str(seed) # Evita variabilidad en hashing de Python
# Fijar la semilla en PyTorch
torch.manual_seed(seed) # Asegura resultados reproducibles en operaciones de PyTorch
if torch.cuda.is_available(): # Si hay GPU disponible
  torch.cuda.manual_seed(seed) # Fija la semilla para la GPU
  torch.backends.cudnn.deterministic = True # Hace las operaciones de CUDNN determinísticas
  torch.backends.cudnn.benchmark = False # Desactiva optimizaciones de CUDNN para evitar variabilidad

def set_global_seed(seed: int):
  random.seed(seed)
  np.random.seed(seed)
  os.environ["PYTHONHASHSEED"] = str(seed)


Usando dispositivo: cpu


### Algoritmos: SARSA y Q-Learning

Ambos algoritmos pertenecen a la familia de **Diferencias Temporales (TD)**. Aprenden paso a paso actualizando sus estimaciones basándose en el error TD (diferencia entre el objetivo estimado y el valor actual), haciendo *bootstrapping*.

#### SARSA (On-Policy)
Aprende el valor de la política que el agente está siguiendo actualmente (incluyendo la exploración $\epsilon$-greedy). Su regla de actualización es:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]$$
* **Característica clave:** Al considerar la acción $A_{t+1}$ que *realmente* va a tomar, es más conservador frente a riesgos si la política es estocástica (como $\epsilon$-greedy).

#### Q-Learning (Off-Policy)
Aprende el valor de la **política óptima**, independientemente de la política exploratoria que esté ejecutando el agente. Su regla de actualización usa el operador $\max$:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$
* **Característica clave:** Es más "optimista" y converge directamente a $Q^*$, asumiendo que en el futuro no cometerá errores de exploración, lo que le permite trazar la ruta más corta posible.

## Utilidades: Entorno, política epsilon-greedy y métricas


In [2]:
# ─── Hiperparámetros globales ─────────────────────────────────────────────────
NUM_EPISODES = 6000
GAMMA   = 0.99
ALPHA   = 0.1
EPSILON = 0.1
ENV_NAME = "FrozenLake-v1"


def make_frozenlake(map_name="4x4", is_slippery=False, seed=seed):
    env = gym.make(ENV_NAME, map_name=map_name, is_slippery=is_slippery)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env

@dataclass
class TrainStats:
    returns: list
    lengths: list
    successes: list


def plot_results(stats_dict, title, window=500):
    """
    Genera dos gráficas comparativas con líneas de tendencia desde el inicio.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    colors = ['steelblue', 'darkorange']

    def get_trendline(data, window_size):
        # Calcula media móvil pero permite crecer desde el inicio
        trend = np.zeros_like(data, dtype=float)
        for i in range(len(data)):
            start_idx = max(0, i - window_size + 1)
            trend[i] = np.mean(data[start_idx : i + 1])
        return trend

    for i, (label, st) in enumerate(stats_dict.items()):
        color = colors[i % len(colors)]
        rewards = np.array(st.returns)
        lengths = np.array(st.lengths)
        episodes = np.arange(len(rewards))

        # --- Gráfica 1: Recompensa ---
        avg_rewards = get_trendline(rewards, window)
        axes[0].plot(episodes, avg_rewards, label=label, color=color, alpha=0.9)

        # --- Gráfica 2: Longitud ---
        # Datos crudos suaves
        axes[1].plot(episodes, lengths, color=color, alpha=0.15)
        # Tendencia desde el principio
        avg_lengths = get_trendline(lengths, window)
        axes[1].plot(episodes, avg_lengths, label=f"{label} (trend)", color=color, linewidth=2)

    axes[0].set_title("Recompensa Promedio (Tendencia)")
    axes[0].set_xlabel("Episodio")
    axes[0].set_ylabel("Recompensa Media")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title("Longitud del episodio (Tendencia)")
    axes[1].set_xlabel("Episodio")
    axes[1].set_ylabel("Pasos")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def evaluate_policy(env, policy_vector, seed=None):
    """Ejecuta la política greedy determinista y devuelve acciones + recompensa."""
    state, info = env.reset(seed=seed)
    done = False
    total_reward = 0.0
    action_log = []

    while not done:
        action = int(policy_vector[state])
        action_log.append(action)

        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward

    return action_log, total_reward

## Implementación tabular: SARSA y Q-Learning

Diseño:
- Usamos una Q-table `Q[s, a]`.
- Entrenamiento episódico estándar de Gymnasium.
- Guardamos métricas por episodio:
  - retorno total en FrozenLake es 0 o 1,
  - longitud del episodio,
  - éxito (1 si llega a la meta).



In [3]:
class QLearningAgent:
    def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=0.1):
        """Inicializa todo lo necesario para el aprendizaje."""
        self.n_actions = env.action_space.n
        self.n_states = env.observation_space.n
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((self.n_states, self.n_actions), dtype=np.float64)

    def get_action(self, state) -> int:
        """Indicará qué acción realizar de acuerdo al estado bajo política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)

        qvals = self.Q[state]
        max_q = np.max(qvals)
        candidates = np.flatnonzero(qvals == max_q)
        return int(np.random.choice(candidates))

    def update(self, obs, action, next_obs, reward, terminated, truncated, info):
        """Aplica el algoritmo Q-Learning (Off-Policy)."""
        done = terminated or truncated

        best_next = 0.0 if done else np.max(self.Q[next_obs])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.Q[obs, action]

        self.Q[obs, action] += self.alpha * td_error

class SarsaAgent:
    def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=0.1):
        """Inicializa todo lo necesario para el aprendizaje."""
        self.n_actions = env.action_space.n
        self.n_states = env.observation_space.n
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((self.n_states, self.n_actions), dtype=np.float64)
        self.next_action = None

    def _choose_epsilon_greedy(self, state) -> int:
        """Lógica interna para elegir acción epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        qvals = self.Q[state]
        max_q = np.max(qvals)
        candidates = np.flatnonzero(qvals == max_q)
        return int(np.random.choice(candidates))

    def get_action(self, state) -> int:
        """Retorna la acción elegida. Usa la calculada en el update previo si existe."""
        if self.next_action is not None:
            action = self.next_action
            self.next_action = None
            return action
        return self._choose_epsilon_greedy(state)

    def update(self, obs, action, next_obs, reward, terminated, truncated, info):
        """Aplica el algoritmo SARSA (On-Policy)."""
        done = terminated or truncated

        if not done:
            self.next_action = self._choose_epsilon_greedy(next_obs)
            next_q = self.Q[next_obs, self.next_action]
        else:
            self.next_action = None
            next_q = 0.0

        td_target = reward + self.gamma * next_q
        td_error = td_target - self.Q[obs, action]

        self.Q[obs, action] += self.alpha * td_error

def train_agent(env, agent, n_episodes=5000, seed=seed, desc="Entrenando"):
    """Esquema general del aprendizaje episódico."""
    stats = TrainStats(returns=[], lengths=[], successes=[])
    set_global_seed(seed)

    for ep in tqdm(range(n_episodes), desc=desc, leave=False):
        obs, info = env.reset(seed=seed + ep)
        done = False
        total_return = 0.0
        t = 0

        while not done:
            action = agent.get_action(obs)
            next_obs, reward, terminated, truncated, info = env.step(action)
            agent.update(obs, action, next_obs, reward, terminated, truncated, info)

            done = terminated or truncated
            obs = next_obs
            total_return += reward
            t += 1

        stats.returns.append(total_return)
        stats.lengths.append(t)
        stats.successes.append(1.0 if total_return > 0 else 0.0)

    return agent.Q, stats

## visualizar política greedy (y un mapa de valores)

Representamos la politica con flechas de forma sencilla

In [4]:

ACTION_SYMBOLS = {0: '←', 1: '↓', 2: '→', 3: '↑'}


def print_policy(Q, label, map_name="4x4"):
    """
    Muestra la política greedy derivada de Q en formato de cuadrícula con bordes.
    Soporta mapas 4×4 y 8×8 de FrozenLake.
    """
    if map_name == "4x4":
        n = 4
        grid = ['S','F','F','F',
                'F','H','F','H',
                'F','F','F','H',
                'H','F','F','G']
    else:
        # Mapa 8×8 estándar de FrozenLake
        n = 8
        desc = gym.make("FrozenLake-v1", map_name="8x8").unwrapped.desc.astype(str)
        grid = [desc[r, c] for r in range(n) for c in range(n)]

    border_w = n * 3 + 1
    print(f"\nPolítica aprendida — {label}")
    print("┌" + "─" * border_w + "┐")
    for row in range(n):
        line = "│ "
        for col in range(n):
            state = row * n + col
            cell = grid[state]
            if cell in ('H', 'G'):
                line += f"{cell}  "
            else:
                best_action = int(np.argmax(Q[state]))
                line += f"{ACTION_SYMBOLS[best_action]}  "
        print(line + "│")
    print("└" + "─" * border_w + "┘")
    print("  (←:0  ↓:1  →:2  ↑:3)")


def plot_state_values(Q, title="Max Q(s,a)", map_name="4x4"):
    """Mapa de calor de max_a Q(s,a) para cada estado."""
    n = 4 if map_name == "4x4" else 8
    V = np.max(Q, axis=1).reshape(n, n)

    plt.figure(figsize=(5, 4))
    plt.imshow(V, cmap='viridis')
    plt.colorbar()
    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.title(title)
    plt.tight_layout()
    plt.show()

## 5) Estudio 1: FrozenLake 4x4 (determinista)

Hipótesis razonable:
- En versión determinista (`is_slippery=False`) ambos algoritmos deberían aprender rápido.
- Al ser un mapa pequeño (16 estados), la señal de reward se propaga eficientemente con TD(0).


In [ ]:

def run_comparison(map_name="4x4", is_slippery=False, n_episodes=8000,
                   alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, seed=seed):
    """
    Entrena SARSA y Q-Learning, muestra gráficas comparativas y políticas.
    Devuelve las Q-tables y estadísticas de ambos agentes.
    """
    env = make_frozenlake(map_name=map_name, is_slippery=is_slippery, seed=seed)

    # Instanciamos y entrenamos SARSA
    sarsa_agent = SarsaAgent(env, alpha=alpha, gamma=gamma, epsilon=epsilon)
    Qs, st_sarsa = train_agent(env, sarsa_agent, n_episodes=n_episodes, seed=seed, desc="SARSA")

    # Instanciamos y entrenamos Q-Learning
    q_agent = QLearningAgent(env, alpha=alpha, gamma=gamma, epsilon=epsilon)
    Qq, st_q = train_agent(env, q_agent, n_episodes=n_episodes, seed=seed, desc="Q-Learning")

    # ─── Gráficas comparativas (recompensa + longitud) ────────────────────────
    label = f"FrozenLake {map_name} | slippery={is_slippery}"
    plot_results({"SARSA": st_sarsa, "Q-Learning": st_q}, title=label)

    # ─── Tasa de éxito y longitud media (últimos 500 episodios) ───────────────
    EVAL_WINDOW = 500
    print(f"\nTasa de éxito — últimos {EVAL_WINDOW} episodios:")
    print(f"  SARSA      : {np.mean(st_sarsa.successes[-EVAL_WINDOW:]):.2%}")
    print(f"  Q-Learning : {np.mean(st_q.successes[-EVAL_WINDOW:]):.2%}")

    print(f"\nLongitud media — últimos {EVAL_WINDOW} episodios:")
    print(f"  SARSA      : {np.mean(st_sarsa.lengths[-EVAL_WINDOW:]):.1f} pasos")
    print(f"  Q-Learning : {np.mean(st_q.lengths[-EVAL_WINDOW:]):.1f} pasos")

    # ─── Políticas greedy + mapa de valores ────────────────────────────────────
    print_policy(Qs, label="SARSA", map_name=map_name)
    plot_state_values(Qs, title=f"SARSA — max Q(s,a) [{map_name}]", map_name=map_name)

    print_policy(Qq, label="Q-Learning", map_name=map_name)
    plot_state_values(Qq, title=f"Q-Learning — max Q(s,a) [{map_name}]", map_name=map_name)

    # ─── Evaluación de la política greedy ──────────────────────────────────────
    env_eval = make_frozenlake(map_name=map_name, is_slippery=is_slippery, seed=seed)
    policy_s = np.argmax(Qs, axis=1)
    policy_q = np.argmax(Qq, axis=1)

    actions_s, reward_s = evaluate_policy(env_eval, policy_s, seed=seed)
    actions_q, reward_q = evaluate_policy(env_eval, policy_q, seed=seed)

    print("\nEvaluación greedy pura:")
    print(f"  SARSA      : acciones={actions_s} → recompensa={reward_s}")
    print(f"  Q-Learning : acciones={actions_q} → recompensa={reward_q}")

    return (Qs, st_sarsa), (Qq, st_q)


# Ejecuta el estudio en 4x4 determinista
(Q_sarsa_4x4, _), (Q_qlearn_4x4, _) = run_comparison(
    map_name="4x4", is_slippery=False, n_episodes=6000
)

### Análisis Resultados en FrozenLake 4x4

En el entorno **4x4**, los resultados muestran una convergencia casi idéntica entre SARSA y Q-Learning, alcanzando ambos una tasa de éxito superior al 85%. Ambos algoritmos identifican la ruta óptima prácticamente al mismo tiempo, sin que la estrategia de actualización marque una diferencia.

La ligera diferencia en pasos (8.0 de SARSA frente a 6.3 de Q-Learning en los últimos episodios) se explica por su naturaleza técnica. **Q-Learning** ignora los movimientos aleatorios de exploración para actualizar sus valores, lo que le permite fijar el camino más corto de forma más agresiva. **SARSA**, al ser *on-policy*, incorpora esos pequeños desvíos exploratorios en su aprendizaje, lo que lo hace ver un poco más conservador en la gráfica de pasos.

En conclusión, los datos del 4x4 confirman que ambos métodos son altamente eficaces para problemas de baja complejidad. La distinción entre aprender del comportamiento real o del ideal es sutil aquí, ya que el riesgo de error es bajo y la ruta óptima es fácil de recuperar incluso tras una acción aleatoria, permitiendo que ambos agentes dominen el entorno con una estabilidad muy similar.

In [ ]:
# Estudio 2: FrozenLake 8x8 determinista con hiperparámetros fijos
(Q_sarsa_8x8, _), (Q_qlearn_8x8, _) = run_comparison(
    map_name="8x8", is_slippery=False, n_episodes=6000
)


### Análisis Resultados en FrozenLake 8x8

En el entorno **8x8**, la complejidad del problema aumenta considerablemente en comparación con el mapa **4x4**. Como se observó en el caso **4x4**, ambos algoritmos, SARSA y Q-Learning, convergen a políticas exitosas. Sin embargo, en este entorno más grande, se aprecian diferencias más marcadas y una convergencia más lenta en términos de episodios. Las tasas de éxito, aunque altas (alrededor del 90%), se logran después de un mayor número de interacciones, evidenciando el desafío adicional que presenta un espacio de estados ampliado. La propagación de la señal de recompensa es menos eficiente, lo que resulta en curvas de aprendizaje más pronunciadas y mesetas iniciales más extensas, un fenómeno apenas perceptible en el mapa 4x4 pero muy evidente aquí.

En cuanto a la distinción entre **SARSA** (on-policy) y **Q-Learning** (off-policy), las observaciones del entorno **4x4** aumentan en el mapa **8x8**. Q-Learning, al actualizarse con el valor máximo de la siguiente acción, tiende a ser más "optimista" y, en promedio, encuentra rutas óptimas con menos pasos. Su naturaleza off-policy le permite aprender de la política óptima independientemente de la exploración actual, lo que lo hace ligeramente más eficiente en términos de longitud de episodio. Por otro lado, SARSA, al considerar la acción $A_{t+1}$ que realmente tomará (incluyendo la exploración), mantiene un enfoque más conservador. Aunque esto puede llevar a rutas ligeramente más largas o una convergencia más lenta en entornos deterministas, como se insinuó en el 4x4, se espera que le otorgue una mayor robustez en entornos estocásticos o con penalizaciones por error más severas, ya que su política Q refleja con mayor fidelidad los riesgos de la exploración.

##Conclusion

Tras el desarrollo de estos experimentos, hemos podido extraer varias conclusiones al comparar el comportamiento de ambos algoritmos. Por un lado, observando la velocidad de convergencia en los entornos deterministas, se nota que Q-Learning tiene una propagacion de valores hacia atras ligeramente mas rapida y agresiva. Esto es basicamente gracias a que actualiza sus valores usando el operador max, asumiendo que siempre tomara la mejor decision posible en el siguiente paso. Esto se refleja muy bien en las graficas de entrenamiento, donde hemos podido ver como Q-Learning logra reducir la longitud de los episodios en las etapas intermedias bastante antes que SARSA, estabilizandose mas rapido en la ruta mas corta hacia el objetivo.

Por otro lado, al comparar el enfoque On-Policy frente al Off-Policy, hemos podido comprobar como afecta realmente la exploracion al aprendizaje de cada agente. SARSA evalua la politica epsilon-greedy real que el agente esta ejecutando en ese momento, la cual incluye siempre esa pequeña probabilidad del 10% de tomar una accion totalmente aleatoria. Debido a esto, el agente aprende valores Q bastante mas bajos en los estados que son peligrosos o que estan directamente pegados a los agujeros, ya que es consciente de que puede caerse por culpa de su propia exploracion. Aunque en un entorno determinista sin resbalones esta diferencia en la ruta optima acaba siendo muy sutil visualmente, en los numeros esto hace que SARSA sea un algoritmo mucho mas conservador y seguro frente a Q-Learning, que a veces peca de ser demasiado optimista e ignora los riesgos reales de explorar cerca del peligro.

Finalmente, respecto a la escalabilidad de estos metodos tabulares, pasar del mapa pequeño de 16 estados al grande de 64 estados me demostro lo mucho que sufren estos algoritmos cuando la señal de recompensa es tan escasa. El tiempo necesario para salir de la meseta inicial de recompensa nula crece casi de forma exponencial en el mapa grande. Esto sucede sobre todo por la enorme dificultad de la exploracion combinatoria en los primeros episodios, donde el agente se pasa muchisimos pasos dando vueltas al azar antes de tropezar por primera vez con el objetivo final. Esto me deja clarisimo que aunque el metodo tabular funciona perfecto para problemas pequeños, sufre demasiado cuando el camino hacia la meta se alarga y depende puramente del azar inicial.